In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import numpy as np
import shutil

In [ ]:
def calculate_global_max():
    """
    Iterates through Train, Val, and Test splits in the '05_mixed' stage
    to find the absolute maximum amplitude across the entire project.
    """
    global_max = 0.0
    print(f"--- Phase 1: Calculating Global Maximum from {STAGE_MIXED} ---")

    for split in SPLITS:
        # Construct path: .../05_mixed/{split}/mixed_dataset.npy
        file_path = get_unet_path(stage=STAGE_MIXED, split=split, signal=MIXED_DATASET)
        
        if not file_path.exists():
            print(f"Warning: File not found at {file_path}")
            continue

        print(f"Scanning {split.upper()} set...")
        
        data = np.load(file_path)
        
        # Calculate local max magnitude (abs handles complex I/Q correctly)
        local_max = np.max(np.abs(data))
        
        if local_max > global_max:
            global_max = local_max
            
        print(f"  -> Local Max: {local_max:.6f}")
        
        # Free memory
        del data

    print(f"\nFinal Global Maximum: {global_max:.6f}")
    return global_max

In [ ]:
def apply_scaling(global_max):
    """
    Loads data from '05_mixed', divides by global_max, and saves to '06_scaled'.
    Also copies the metadata CSVs to keep the dataset complete.
    """
    print(f"\n--- Phase 2: Applying Scaling & Saving to {STAGE_SCALED} ---")
    
    for split in SPLITS:
        # Define Input Paths (05_mixed)
        input_data_path = get_unet_path(stage=STAGE_MIXED, split=split, signal=MIXED_DATASET)

        input_meta_path = get_unet_path(stage=STAGE_MIXED, split=split, signal=MIXED_METADATA, extension="csv")

        # Define Output Paths (06_scaled)
        output_data_path = get_unet_path(stage=STAGE_SCALED, split=split, signal=MIXED_DATASET)
        output_meta_path = get_unet_path(stage=STAGE_SCALED, split=split, signal=MIXED_METADATA, extension="csv")
        
        if input_data_path.exists():
            print(f"Processing {split}...")
            
            # Load Mixed Data
            data = np.load(input_data_path)
            
            # Apply Scaling
            # x_new = x / global_max (Preserves SINR/Energy Ratios)
            scaled_data = data / global_max
            
            # Save Scaled Data
            np.save(output_data_path, scaled_data)
            print(f"  -> Saved Scaled Data: {output_data_path}")
            
            # Copy Metadata (Metadata doesn't change, just the pixel values)
            if input_meta_path.exists():
                shutil.copy(input_meta_path, output_meta_path)
                print(f"  -> Copied Metadata:   {output_meta_path}")
            else:
                print(f"  -> Warning: No metadata found at {input_meta_path}")
                
            del data
            del scaled_data
        else:
            print(f"Skipping {split} (Input not found)")

In [ ]:
master_max = calculate_global_max()

# Save the Artifact (Required for Inference De-normalization)
with open(SCALING_FACTORS_FILE, "w") as f:
    f.write(str(master_max))
print(f"Saved scaling artifact to: {SCALING_FACTORS_FILE}")

# Scale and Move Files
apply_scaling(master_max)

print("Global Scaling Complete.")